# Day 01 — What is Apache Spark? A Gentle Introduction
**Book:** Spark: The Definitive Guide | **Chapters:** 1 & 2  
**Time estimate:** 60–75 min

---

## What you'll learn today
- What Spark is and why it was built
- The core execution model: Driver, Executors, Partitions
- The single most important concept: **Transformations vs Actions (lazy evaluation)**
- Narrow vs Wide transformations and why they matter for performance
- How to read the Spark UI

> **How to use this notebook:** Run each cell with `Shift+Enter`. Every code cell has a comment explaining *why*, not just *what*. Don't skip the markdown cells — they contain the mental models.


## 0. Imports & SparkSession setup

Always the first cell in every notebook. We define a reusable `get_spark()` factory so every day follows the same pattern.


In [2]:
from pyspark.sql import SparkSession

def get_spark(app_name: str = "spark_study", shuffle_partitions: int = 4) -> SparkSession:
    """
    Standard SparkSession factory used throughout this series.
    
    shuffle_partitions=4 is sensible for local development on a laptop.
    The default of 200 would create 200 tiny tasks for every groupBy/join,
    which wastes overhead when you have 8 cores, not 200.
    """
    return (
        SparkSession.builder
        .appName(app_name)
        .master("local[*]")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.shuffle.partitions", str(shuffle_partitions))
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )

spark = get_spark("day01_spark_intro")
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ Spark {spark.version} ready")
print(f"   App name : {spark.sparkContext.appName}")
print(f"   Master   : {spark.sparkContext.master}")
print(f"   Cores    : {spark.sparkContext.defaultParallelism}")
print(f"   Spark UI : http://localhost:4040")


✅ Spark 4.1.2 ready
   App name : day01_spark_intro
   Master   : local[*]
   Cores    : 15
   Spark UI : http://localhost:4040


---
## 1. What is Apache Spark and why does it exist?

**The problem it solved:** Before Spark (2009), the dominant tool for large-scale data processing was **Hadoop MapReduce**. MapReduce had one fatal flaw: it wrote intermediate results to disk after every step. A 10-step job read and wrote disk 10 times. For iterative algorithms (machine learning, graph processing), this was catastrophically slow.

**Spark's insight:** Keep intermediate data **in memory** (RAM) across operations. A 10-step job only touches disk at the start and end.

```
MapReduce:  Read disk → Process → Write disk → Read disk → Process → Write disk → ...
Spark:      Read disk → [in-memory pipeline] → Write disk
```

**The result:** Spark is typically **10–100× faster** than MapReduce for iterative workloads.

### Spark's unified stack

| Module | Purpose |
|--------|---------|
| **Spark SQL / DataFrames** | Structured data, SQL queries |
| **Structured Streaming** | Real-time stream processing |
| **MLlib** | Scalable machine learning |
| **GraphX / GraphFrames** | Graph computation |

All of these run on the same engine. One tool, one cluster, all workloads.


---
## 2. The Cluster Architecture

Understanding this diagram is fundamental to debugging performance.

```
┌─────────────────────────────────────────┐
│              DRIVER (your laptop)       │
│                                         │
│  SparkSession / SparkContext            │
│  ┌─────────────────────────────────┐    │
│  │ Your Python code                │    │
│  │ → Catalyst Optimizer            │    │
│  │ → DAG Scheduler                 │    │
│  │ → Task Scheduler                │    │
│  └─────────────────────────────────┘    │
└──────────────┬──────────────────────────┘
               │ sends tasks over network
    ┌──────────┼──────────┐
    ▼          ▼          ▼
┌────────┐ ┌────────┐ ┌────────┐
│Executor│ │Executor│ │Executor│  ← JVM processes
│        │ │        │ │        │
│[part 0]│ │[part 1]│ │[part 2]│  ← each holds partitions
│[part 3]│ │[part 4]│ │[part 5]│
└────────┘ └────────┘ └────────┘
```

**In local mode** (your laptop), Driver and Executors all live in the same JVM process. There's no actual network — but the same code runs unchanged on a 1000-node cluster.

### Key terms

| Term | One-line definition |
|------|-------------------|
| **Driver** | The process running your SparkSession. Orchestrates everything. |
| **Executor** | Worker JVM that runs tasks. Holds partitions in memory/disk. |
| **Partition** | One chunk of your data. Unit of parallelism. Lives on one executor. |
| **Task** | One unit of work = one function applied to one partition. |
| **Stage** | A set of tasks that can run without shuffling data. |
| **Job** | All the stages triggered by one action (e.g., `.count()`). |


In [3]:
# Let's see the architecture in action with a simple DataFrame.
# spark.range(n) creates a DataFrame with a single 'id' column: 0, 1, 2, ... n-1

df = spark.range(1000)

print(f"Type: {type(df)}")
print(f"Number of partitions: {df.rdd.getNumPartitions()}")
print(f"Schema: {df.schema}")

# show() displays the first n rows. It is an ACTION — triggers computation.
df.show(5)


Type: <class 'pyspark.sql.classic.dataframe.DataFrame'>
Number of partitions: 15
Schema: StructType([StructField('id', LongType(), False)])
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+
only showing top 5 rows


---
## 3. The Most Important Concept: Transformations vs Actions

This is the concept that surprises every Spark newcomer and explains most performance behaviour.

### Transformations (lazy — nothing runs)
Return a new DataFrame. Spark records the *instruction* but does not execute it.
```python
df2 = df.filter(df.id > 100)    # recorded, not executed
df3 = df2.withColumn("x", ...)  # recorded, not executed
```

### Actions (eager — triggers execution)
Compute a result and return it (to driver or to storage).
```python
df3.count()          # NOW Spark executes filter + withColumn + count
df3.show()           # NOW Spark executes and displays
df3.collect()        # NOW Spark executes and pulls ALL data to driver (⚠️)
df3.write.parquet()  # NOW Spark executes and writes to disk
```

### Why laziness?
Spark batches your transformations and hands them to the **Catalyst Optimizer**, which may:
- Reorder operations (push filters earlier)
- Eliminate unnecessary columns
- Combine multiple steps into one physical operation

You write readable code; Spark executes an optimised plan.


In [4]:
from pyspark.sql import functions as F

# Demonstrating lazy evaluation — nothing runs until an action

# Step 1: Create some data
data = [
    ("Alice",   "Engineering", 95_000),
    ("Bob",     "Marketing",   72_000),
    ("Carol",   "Engineering", 88_000),
    ("Dave",    "Marketing",   81_000),
    ("Eve",     "Engineering", 102_000),
    ("Frank",   "HR",          68_000),
    ("Grace",   "HR",          74_000),
]
employees = spark.createDataFrame(data, schema=["name", "department", "salary"])

# Step 2: Define transformations (NOTHING runs here — just building a plan)
high_earners = employees.filter(employees.salary > 80_000)         # transformation
with_bonus   = high_earners.withColumn("bonus", F.col("salary") * 0.1)  # transformation
selected     = with_bonus.select("name", "department", "salary", "bonus")  # transformation

print("Transformations defined. No computation has happened yet.")
print(f"Type of 'selected': {type(selected)}")  # still a DataFrame (a plan)


Transformations defined. No computation has happened yet.
Type of 'selected': <class 'pyspark.sql.classic.dataframe.DataFrame'>


In [5]:
# Step 3: An ACTION triggers the full plan above
print("Calling .show() — THIS triggers execution:")
selected.show()

# The execution plan shows exactly what Spark will do
# Look for: Filter, Project (= select), Scan
print("\nExecution plan:")
selected.explain()


Calling .show() — THIS triggers execution:
+-----+-----------+------+-------+
| name| department|salary|  bonus|
+-----+-----------+------+-------+
|Alice|Engineering| 95000| 9500.0|
|Carol|Engineering| 88000| 8800.0|
| Dave|  Marketing| 81000| 8100.0|
|  Eve|Engineering|102000|10200.0|
+-----+-----------+------+-------+


Execution plan:
== Physical Plan ==
*(1) Project [name#5, department#6, salary#7L, (cast(salary#7L as double) * 0.1) AS bonus#8]
+- *(1) Filter (isnotnull(salary#7L) AND (salary#7L > 80000))
   +- *(1) Scan ExistingRDD[name#5,department#6,salary#7L]




In [6]:
# Another action on the SAME transformations — Spark re-executes from scratch
# (unless you cache — covered in Part IV)
count = selected.count()
print(f"Count of high earners: {count}")

# .collect() brings ALL rows to the driver as a Python list
# Fine for small DataFrames, dangerous for large ones
rows = selected.collect()
print(f"\nType returned by collect(): {type(rows)}")
print(f"First row: {rows[0]}")
print(f"Access by column name: {rows[0]['name']}, salary={rows[0]['salary']}")


Count of high earners: 4

Type returned by collect(): <class 'list'>
First row: Row(name='Alice', department='Engineering', salary=95000, bonus=9500.0)
Access by column name: Alice, salary=95000


---
## 4. Narrow vs Wide Transformations

This distinction explains why some operations are fast and others cause a performance bottleneck.

### Narrow transformation
Each output partition depends on **at most one** input partition.  
No data moves between executors. Each executor works independently.

```
Partition 0 ──filter──▶ Partition 0'
Partition 1 ──filter──▶ Partition 1'
Partition 2 ──filter──▶ Partition 2'
```

**Examples:** `filter`, `select`, `withColumn`, `map`, `union`, `coalesce`

### Wide transformation (shuffle)
Each output partition may depend on **multiple** input partitions.  
Spark must **shuffle** data across executors — the most expensive operation.  
Shuffles create **stage boundaries** in the DAG.

```
Partition 0 ──┐
Partition 1 ──┼──groupBy──▶ Partition 0' (Engineering rows from all inputs)
Partition 2 ──┘            Partition 1' (Marketing rows from all inputs)
```

**Examples:** `groupBy`, `orderBy`, `join`, `repartition`, `distinct`, `pivot`

> 💡 **Performance tip:** Minimise shuffles. Push narrow transformations (especially `filter`) as early as possible to reduce the data that needs to be shuffled.


In [7]:
# Narrow transformation: filter — each partition filtered independently, no shuffle
senior = employees.filter(employees.salary > 80_000)

# Wide transformation: groupBy + agg — requires shuffling all rows
# so matching department keys end up on the same executor
dept_stats = (
    employees
    .groupBy("department")
    .agg(
        F.avg("salary").alias("avg_salary"),
        F.max("salary").alias("max_salary"),
        F.count("*").alias("headcount")
    )
    .orderBy("avg_salary", ascending=False)
)

dept_stats.show()


+-----------+----------+----------+---------+
| department|avg_salary|max_salary|headcount|
+-----------+----------+----------+---------+
|Engineering|   95000.0|    102000|        3|
|  Marketing|   76500.0|     81000|        2|
|         HR|   71000.0|     74000|        2|
+-----------+----------+----------+---------+



In [8]:
# The explain() plan reveals stage boundaries
# Look for the keyword 'Exchange' — that's a shuffle (= stage boundary)
print("=== Plan for groupBy + agg (wide transformation) ===")
dept_stats.explain(mode="formatted")  # 'formatted' gives a cleaner tree in Spark 3+


=== Plan for groupBy + agg (wide transformation) ===
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [3]: [name#5, department#6, salary#7L]
Arguments: [name#5, department#6, salary#7L], MapPartitionsRDD[14] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [2]: [department#6, salary#7L]
Input [3]: [name#5, department#6, salary#7L]

(3) HashAggregate
Input [2]: [department#6, salary#7L]
Keys [1]: [department#6]
Functions [3]: [partial_avg(salary#7L), partial_max(salary#7L), partial_count(1)]
Aggregate Attributes [4]: [sum#43, count#44L, max#45L, count#46L]
Results [5]: [department#6, sum#47, count#48L, max#49L, count#50L]

(4) Exchange
Input [5]: [department#6, sum#47, count#48L, max#49L, count#50L]
Arguments: h

---
## 5. Partitions — The Unit of Parallelism

A **partition** is a contiguous chunk of your dataset that lives on one executor and is processed by one task. More partitions = more parallelism (up to your number of CPU cores).

### Two key configs

| Config | Controls | Default |
|--------|---------|---------|
| `spark.default.parallelism` | RDD parallelism, initial DataFrame partitions | # cores |
| `spark.sql.shuffle.partitions` | Partitions created after a shuffle | **200** |

The shuffle default of 200 is designed for production clusters with hundreds of cores. On a laptop with 8 cores, 200 shuffle partitions means 200 tiny tasks with scheduling overhead. **Always set this to 2–4× your core count for local work.**

### `repartition` vs `coalesce`

| | `repartition(n)` | `coalesce(n)` |
|-|-----------------|--------------|
| Direction | Can increase or decrease | Can only decrease |
| Shuffle | Always shuffles (full) | No shuffle (merges local partitions) |
| Distribution | Perfectly balanced | May be uneven |
| Use when | You need balanced partitions or more partitions | You just want fewer partitions cheaply |


In [9]:
print(f"Default parallelism (cores): {spark.sparkContext.defaultParallelism}")
print(f"Shuffle partitions config  : {spark.conf.get('spark.sql.shuffle.partitions')}")

# Demonstrate repartition vs coalesce
df = spark.range(10_000)

print(f"\nOriginal partitions: {df.rdd.getNumPartitions()}")

df_8 = df.repartition(8)
print(f"After repartition(8): {df_8.rdd.getNumPartitions()}  ← full shuffle, balanced")

df_3 = df_8.coalesce(3)
print(f"After coalesce(3):   {df_3.rdd.getNumPartitions()}  ← no shuffle, merged")

df_12 = df_3.repartition(12)
print(f"After repartition(12): {df_12.rdd.getNumPartitions()} ← full shuffle again")


Default parallelism (cores): 15
Shuffle partitions config  : 4

Original partitions: 15
After repartition(8): 8  ← full shuffle, balanced
After coalesce(3):   3  ← no shuffle, merged
After repartition(12): 12 ← full shuffle again


In [10]:
# Inspect rows per partition using mapPartitionsWithIndex (RDD API, covered in Part III)
# This is a useful debugging technique

def count_rows_in_partition(index, iterator):
    count = sum(1 for _ in iterator)
    yield (index, count)

print("Row distribution across partitions (repartition — balanced):")
repartitioned = employees.repartition(4)
dist = repartitioned.rdd.mapPartitionsWithIndex(count_rows_in_partition).collect()
for partition_id, count in sorted(dist):
    bar = "█" * count
    print(f"  Partition {partition_id}: {count:3d} rows  {bar}")


Row distribution across partitions (repartition — balanced):
  Partition 0:   2 rows  ██
  Partition 1:   1 rows  █
  Partition 2:   1 rows  █
  Partition 3:   3 rows  ███


---
## 6. Schemas — Define Them Explicitly

Spark can infer schemas automatically, but for production code (and for learning), always define schemas explicitly. Inference requires Spark to scan the data, which costs an extra pass.


In [11]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Explicit schema definition — best practice
employee_schema = StructType([
    StructField("name",       StringType(),  nullable=False),
    StructField("department", StringType(),  nullable=False),
    StructField("salary",     IntegerType(), nullable=False),
])

typed_employees = spark.createDataFrame(data, schema=employee_schema)

print("Schema (defined explicitly):")
typed_employees.printSchema()

# You can also write schemas as DDL strings — concise and readable
ddl_schema = "name STRING NOT NULL, department STRING NOT NULL, salary INT NOT NULL"
typed2 = spark.createDataFrame(data, schema=ddl_schema)
print("Schema from DDL string:")
typed2.printSchema()


Schema (defined explicitly):
root
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- salary: integer (nullable = false)

Schema from DDL string:
root
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- salary: integer (nullable = false)



---
## 7. The Spark UI — Your Debugging Command Centre

While this notebook is running, open **http://localhost:4040** in your browser.

### What each tab shows you

| Tab | What to look for |
|-----|-----------------|
| **Jobs** | Each `.count()`, `.show()`, `.write()` = one job. See duration and success. |
| **Stages** | Each stage = one set of tasks between shuffles. Look at shuffle read/write bytes. |
| **Tasks** | Individual task durations. Outliers = data skew (a major perf problem). |
| **SQL / DataFrame** | Visual DAG of your DataFrame operations. Filters show as pushdowns. |
| **Storage** | What's cached (`.cache()` / `.persist()`). |
| **Environment** | All Spark config values. |
| **Executors** | Per-executor memory usage and GC time. |

Run the cell below, then immediately check the UI's **SQL** tab to see the query plan visualised as a DAG.


In [12]:
# Run a multi-step operation and explore it in the Spark UI

result = (
    employees
    .filter(F.col("salary") > 70_000)                          # narrow
    .withColumn("tax", F.col("salary") * 0.3)                  # narrow
    .groupBy("department")                                      # wide (shuffle here)
    .agg(
        F.round(F.avg("salary"), 2).alias("avg_salary"),
        F.round(F.sum("tax"), 2).alias("total_tax_burden"),
        F.count("*").alias("headcount")
    )
    .orderBy("avg_salary", ascending=False)                     # wide (another shuffle)
)

result.show()

# After running this cell, check http://localhost:4040 → SQL tab
# You'll see: Scan → Filter → Project → HashAggregate → Exchange → Sort
# Each Exchange = a shuffle = a stage boundary
print("\n✅ Check http://localhost:4040 → SQL tab for the visual DAG")


+-----------+----------+----------------+---------+
| department|avg_salary|total_tax_burden|headcount|
+-----------+----------+----------------+---------+
|Engineering|   95000.0|         85500.0|        3|
|  Marketing|   76500.0|         45900.0|        2|
|         HR|   74000.0|         22200.0|        1|
+-----------+----------+----------------+---------+


✅ Check http://localhost:4040 → SQL tab for the visual DAG


---
## Summary

| Concept | Key takeaway |
|---------|-------------|
| **Driver vs Executor** | Driver plans; Executors run tasks on partitions |
| **Transformations** | Lazy — build a plan, nothing executes |
| **Actions** | Eager — trigger execution of the full plan |
| **Narrow** | No shuffle; fast; `filter`, `select`, `withColumn` |
| **Wide** | Shuffle required; creates stage boundary; `groupBy`, `join`, `orderBy` |
| **Partitions** | More = more parallelism. Set shuffle partitions to ~2–4× cores locally |
| **Schemas** | Always define explicitly in production code |
| **Spark UI** | localhost:4040 — your primary debugging tool |

---

## Exercises

Move on to `exercises.ipynb` to test your understanding.
